# 00 - Supervised fine-tuning

Fine-tune Qwen3-0.6B for two epochs on 2,048 GSM8K worked solutions and save one checkpoint for every RL notebook. The official test split remains held out.

The completed checkpoint is hosted privately at `ppbhatt500/rl-ablations-sft-2026-07-17`. The downstream notebooks load it directly with `HF_TOKEN`, so this notebook only needs to run when intentionally retraining the shared SFT base.

## Environment setup

Run this once from the repository root before opening the notebook:

```bash
cd "$(git rev-parse --show-toplevel)"
uv sync
uv run python -m ipykernel install --user --name rl-ablations --display-name "Python (rl-ablations)"
uv run --no-sync --env-file .env jupyter lab
```

Select the **Python (rl-ablations)** kernel. Dependencies are declared in `pyproject.toml`; do not run separate `uv add` commands inside the notebook.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
expected_python = (project_root / ".venv" / "bin" / "python").resolve()
if Path(sys.executable).resolve() != expected_python:
    raise RuntimeError(
        f"Wrong Jupyter kernel: {sys.executable}. "
        "Select 'Python (rl-ablations)' and restart the notebook."
    )
print(f"Using project environment: {sys.executable}")

Using project environment: /home/ubuntu/rl-ablations/.venv/bin/python3


In [2]:
# Hyperparameters
base_model = "Qwen/Qwen3-0.6B"
dataset_id = "openai/gsm8k"
dataset_config = "main"
dataset_revision = "740312add88f781978c0658806c59bc2815b9866"
seed = 17
train_problems = 2048
eval_problems = 64
num_epochs = 2
max_sequence_length = 1024
train_micro_batch_size = 2
gradient_accumulation_steps = 8
learning_rate = 2e-5
weight_decay = 0.0
warmup_fraction = 0.03
logging_steps = 1
save_path = "./checkpoints/sft_base"
wandb_project = "swe-rl-ablations"
wandb_run_name = "sft-base"

effective_batch_size = train_micro_batch_size * gradient_accumulation_steps
optimizer_steps_per_epoch = train_problems // effective_batch_size
total_optimizer_steps = optimizer_steps_per_epoch * num_epochs
warmup_steps = max(1, round(total_optimizer_steps * warmup_fraction))
eval_steps = optimizer_steps_per_epoch
assert train_problems == 2048 and eval_problems == 64 and num_epochs == 2
assert train_problems % effective_batch_size == 0
print(f"SFT steps: {total_optimizer_steps}; effective batch size: {effective_batch_size}")

SFT steps: 256; effective batch size: 16


In [3]:
from pathlib import Path
import os
import re

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from trl import SFTConfig, SFTTrainer

project_root = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
checkpoint_path = project_root / save_path
os.environ["WANDB_PROJECT"] = wandb_project

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this project.")
try:
    import flash_attn  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Install flash-attn before running the notebook.") from exc
set_seed(seed)

In [4]:
train_rows = load_dataset(dataset_id, dataset_config, split="train", revision=dataset_revision)
eval_rows = load_dataset(dataset_id, dataset_config, split="test", revision=dataset_revision)
if set(train_rows.column_names) != {"question", "answer"}:
    raise RuntimeError(f"Unexpected GSM8K schema: {train_rows.column_names}")
train_rows = train_rows.shuffle(seed=seed).select(range(train_problems))
eval_rows = eval_rows.shuffle(seed=seed).select(range(eval_problems))
print(f"SFT train: {len(train_rows)} | held-out test: {len(eval_rows)}")

SFT train: 2048 | held-out test: 64


In [5]:
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def assistant_solution(answer):
    match = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)\s*$", answer)
    if not match:
        raise ValueError(f"Malformed GSM8K answer: {answer!r}")
    reasoning = re.sub(r"<<[^<>]*>>", "", answer[:match.start()]).strip()
    number = match.group(1)
    return f"{reasoning}\nFinal answer: {number}"


def conversation(row):
    messages = [
        {
            "role": "user",
            "content": "Solve this grade-school math problem. Show your reasoning, then end with `Final answer: <number>`.\n\n" + row["question"],
        },
        {"role": "assistant", "content": assistant_solution(row["answer"])},
    ]
    return {"prompt": messages[:1], "completion": messages[1:]}

train_conversations = train_rows.map(conversation, remove_columns=train_rows.column_names)
eval_conversations = eval_rows.map(conversation, remove_columns=eval_rows.column_names)

Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
model.config.use_cache = False

training_args = SFTConfig(
    output_dir=str(checkpoint_path),
    run_name=wandb_run_name,
    max_length=max_sequence_length,
    completion_only_loss=True,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_micro_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    warmup_steps=warmup_steps,
    logging_steps=logging_steps,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="epoch",
    report_to="wandb",
    bf16=True,
    gradient_checkpointing=True,
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_conversations,
    eval_dataset=eval_conversations,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(str(checkpoint_path))
tokenizer.save_pretrained(str(checkpoint_path))
assert (checkpoint_path / "config.json").exists()
print(f"Saved {checkpoint_path}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/2048 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2048 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2048 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
128,0.570453,0.627900,0.606767,386614.000000,0.834527
256,0.552571,0.628692,0.560261,773228.000000,0.837445


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved /home/ubuntu/rl-ablations/checkpoints/sft_base
